In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

# =====================
# Load Dataset
# =====================

df = pd.read_csv("ml_dataset.csv")

# =====================
# Features / Target
# =====================

X = df.drop("priority", axis=1)

y = df["priority"]

# =====================
# Split
# =====================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =====================
# Model
# =====================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

# =====================
# Evaluate
# =====================

preds = model.predict(X_test)

print("\nAccuracy")

print(
    accuracy_score(
        y_test,
        preds
    )
)

print("\nClassification Report")

print(
    classification_report(
        y_test,
        preds
    )
)

# =====================
# Save
# =====================

joblib.dump(
    model,
    "risk_model.pkl"
)

print("\nModel Saved")


Accuracy
0.998165137614679

Classification Report
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       629
         1.0       1.00      1.00      1.00      1006

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635


Model Saved


In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("impact_dataset.csv")

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Select Features
# ==========================================

features = [
    "event_cause",
    "priority",
    "requires_road_closure",
    "event_type"
]

target = "impact_score"

X = df[features]
y = df[target]

print("\nFeatures:")
print(features)

print("\nTarget:")
print(target)

# ==========================================
# Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

# ==========================================
# Preprocessing
# ==========================================

categorical_features = [
    "event_cause",
    "priority",
    "event_type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# ==========================================
# Model
# ==========================================

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

# ==========================================
# Pipeline
# ==========================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# ==========================================
# Train Model
# ==========================================

print("\nTraining Model...")

pipeline.fit(
    X_train,
    y_train
)

print("Training Completed")

# ==========================================
# Predictions
# ==========================================

predictions = pipeline.predict(
    X_test
)

# ==========================================
# Evaluation
# ==========================================

mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = mse ** 0.5

r2 = r2_score(
    y_test,
    predictions
)

print("\n========== MODEL RESULTS ==========")

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

# ==========================================
# Save Model
# ==========================================

joblib.dump(
    pipeline,
    "traffic_impact_predictor.pkl"
)

print("\nModel Saved Successfully")
print("File: traffic_impact_predictor.pkl")

# ==========================================
# Example Prediction
# ==========================================

sample_event = pd.DataFrame({

    "event_cause": ["construction"],

    "priority": ["High"],

    "requires_road_closure": [True],

    "event_type": ["planned"]

})

predicted_impact = pipeline.predict(
    sample_event
)

print("\nExample Forecast")

print(sample_event)

print(
    f"\nPredicted Traffic Impact Index: "
    f"{predicted_impact[0]:.2f}"
)

# ==========================================
# Risk Classification
# ==========================================

def get_risk(score):

    if score >= 75:
        return "HIGH"

    elif score >= 55:
        return "MEDIUM"

    else:
        return "LOW"

print(
    "Predicted Risk Level:",
    get_risk(predicted_impact[0])
)

Dataset Loaded
(8173, 53)

Features:
['event_cause', 'priority', 'requires_road_closure', 'event_type']

Target:
impact_score

Train Shape: (6538, 4)
Test Shape: (1635, 4)

Training Model...
Training Completed

========== MODEL RESULTS ==========
MAE  : 0.13
MSE  : 0.87
RMSE : 0.93
R²   : 0.9888

Model Saved Successfully
File: traffic_impact_predictor.pkl

Example Forecast
    event_cause priority  requires_road_closure event_type
0  construction     High                   True    planned

Predicted Traffic Impact Index: 83.00
Predicted Risk Level: HIGH


In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("impact_dataset.csv")

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Select Features and Target
# ==========================================
# These are the main operational inputs available
# before or during an incident

feature_cols = [
    "event_cause",
    "priority",
    "requires_road_closure",
    "event_type",
    "junction"
]

target_col = "impact_score"

# Keep only required columns
model_df = df[feature_cols + [target_col]].copy()

# Drop missing target if any
model_df = model_df.dropna(subset=[target_col])

# Fill missing categorical values
for col in ["event_cause", "priority", "event_type", "junction"]:
    model_df[col] = model_df[col].fillna("unknown").astype(str)

# Ensure boolean column is boolean/int
model_df["requires_road_closure"] = model_df["requires_road_closure"].astype(int)

print("\nFeatures:")
print(feature_cols)

print("\nTarget:")
print(target_col)

# ==========================================
# Split Features / Target
# ==========================================

X = model_df[feature_cols]
y = model_df[target_col]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# ==========================================
# Preprocessing
# ==========================================

categorical_features = [
    "event_cause",
    "priority",
    "event_type",
    "junction"
]

numeric_features = [
    "requires_road_closure"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

# ==========================================
# Model
# ==========================================
# RandomForest works well here because:
# - mixed categorical patterns
# - non-linear relationships
# - no scaling required

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# ==========================================
# Train Model
# ==========================================

print("\nTraining Model...")
model.fit(X_train, y_train)
print("Training Completed")

# ==========================================
# Evaluate Model
# ==========================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n========== MODEL RESULTS ==========")
print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

# ==========================================
# Save Model
# ==========================================

model_file = "traffic_impact_predictor.pkl"

with open(model_file, "wb") as f:
    pickle.dump(model, f)

print("\nModel Saved Successfully")
print(f"File: {model_file}")

# ==========================================
# Example Forecast
# ==========================================

example_input = pd.DataFrame([{
    "event_cause": "construction",
    "priority": "high",
    "requires_road_closure": 1,
    "event_type": "planned",
    "junction": "MekhriCircle"
}])

predicted_impact = model.predict(example_input)[0]

print("\nExample Forecast")
print(example_input)

print(f"\nPredicted Traffic Impact Index: {predicted_impact:.2f}")

# ==========================================
# Risk Level Helper
# ==========================================

def get_risk(score):
    if score >= 80:
        return "HIGH"
    elif score >= 60:
        return "MEDIUM"
    else:
        return "LOW"

print(f"Predicted Risk Level: {get_risk(predicted_impact)}")

Dataset Loaded
(8173, 54)

Features:
['event_cause', 'priority', 'requires_road_closure', 'event_type', 'junction']

Target:
impact_score

Train Shape: (6538, 5)
Test Shape : (1635, 5)

Training Model...
Training Completed

========== MODEL RESULTS ==========
MAE  : 0.15
MSE  : 0.70
RMSE : 0.83
R²   : 0.9846

Model Saved Successfully
File: traffic_impact_predictor.pkl

Example Forecast
    event_cause priority  requires_road_closure event_type      junction
0  construction     high                      1    planned  MekhriCircle

Predicted Traffic Impact Index: 71.61
Predicted Risk Level: MEDIUM
